# 🔮 W9: 예지보전 & 이상감지 파이프라인
**hexa-1 — Week 9** | ⏱️ 60분 | 🎯 이상 감지 + 자동 알림

통계적 방법(3σ 규칙)으로 이상 패턴을 자동 감지합니다.

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'], capture_output=True)
!pip install -q pandas matplotlib scipy
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
nanum = next((f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name), None)
if nanum: plt.rcParams['font.family'] = fm.FontProperties(fname=nanum).get_name()
plt.rcParams['axes.unicode_minus'] = False
print('✅ 환경 설정 완료')


## Step 1: 데이터 로드

In [ ]:
import requests, io
url = 'https://raw.githubusercontent.com/Reasonofmoon/hexa-1/main/scripts/oee_sample_data.csv'
try:
    df = pd.read_csv(url, encoding='utf-8-sig')
    print(f'✅ OEE 데이터 로드: {len(df)}행')
except:
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(io.BytesIO(list(up.values())[0]), encoding='utf-8-sig')

date_col   = next((c for c in df.columns if '날짜' in c), df.columns[0])
defect_col = next((c for c in df.columns if '불량' in c and '수' in c), None)
total_col  = next((c for c in df.columns if '생산량' in c), None)
df[date_col] = pd.to_datetime(df[date_col])
df['불량률'] = df[defect_col] / df[total_col] * 100
print(f'사용 컬럼 → 날짜:{date_col}, 불량:{defect_col}, 생산:{total_col}')
df.head(3)


## Step 2: 3σ 이상감지

In [ ]:
# 3-Sigma Rule: 평균 ± 3σ 범위 초과 = 이상
mean = df['불량률'].mean()
std  = df['불량률'].std()
upper = mean + 3 * std
lower = max(0, mean - 3 * std)

df['이상'] = (df['불량률'] > upper) | (df['불량률'] < lower)
anomalies = df[df['이상']]

print(f'📊 정상 범위: {lower:.2f}% ~ {upper:.2f}%')
print(f'⚠️ 이상 감지: {len(anomalies)}건 / 전체 {len(df)}건')
if len(anomalies) > 0:
    print('\n이상 발생일:')
    print(anomalies[[date_col, '불량률']].to_string(index=False))


## Step 3: 시각화 + 알림 메시지 생성

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df[date_col], df['불량률'], 'b-o', label='불량률', markersize=4)
ax.axhline(upper, color='red', linestyle='--', label=f'상한 ({upper:.2f}%)')
ax.axhline(mean, color='green', linestyle='-', label=f'평균 ({mean:.2f}%)')
ax.scatter(anomalies[date_col], anomalies['불량률'],
           color='red', s=100, zorder=5, label=f'이상 ({len(anomalies)}건)')
ax.set_title('예지보전: 불량률 이상감지 (3σ 규칙)')
ax.set_ylabel('불량률 (%)')
ax.legend()
plt.tight_layout()
plt.savefig('anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()

# 알림 메시지
if len(anomalies) > 0:
    print('\n🚨 [예지보전 알림]')
    for _, row in anomalies.iterrows():
        print(f' - {row[date_col].date()}: 불량률 {row["불량률"]:.2f}% (상한 {upper:.2f}% 초과)')
    print('\n→ Slack W07 노트북과 연결해서 자동 알림 발송 가능!')
else:
    print('✅ 이상 없음 — 정상 운영 중')


## Step 4: 결과 저장

In [ ]:
df.to_csv('anomaly_results.csv', index=False, encoding='utf-8-sig')
from google.colab import files
files.download('anomaly_results.csv')
files.download('anomaly_detection.png')
print('✅ 다운로드 완료!')


---
## 🔥 확장 과제
1. 2σ로 바꿔서 더 민감하게 설정
2. 이상 발생 시 W07 Slack 코드와 연결해서 자동 알림
3. 이동평균(rolling mean) 기반 이상감지로 업그레이드